## Bridge Crossing — 4-Queue Simulation

`n` workers, `k` packages on the **right** bank.  
Workers shuttle packages left.  
`time[i] = [leftToRight, pickOld, rightToLeft, putNew]`

**Rules:**
- Bridge fits **one person at a time**  
- **Right → Left has priority** over Left → Right (they carry packages)  
- Among same direction: **higher index = lower efficiency = crosses first**  
- Return the time the **last package is put down** on the left bank

**4 Queues:**

| Queue | Type | Contents |
|-------|------|----------|
| `left_wait` | max-heap (by index) | Ready to cross L→R |
| `right_wait` | max-heap (by index) | Ready to cross R→L |
| `left_work` | min-heap (by time) | Putting package down (putNew) |
| `right_work` | min-heap (by time) | Picking package up (pickOld) |

**Clock** = when the bridge becomes free next.  
At each tick: release finished workers → choose who crosses → advance clock.

In [ ]:
%run _timer.py

In [ ]:
import heapq

def solution(n, k, time):
    # --- Setup -------------------------------------------------------
    # Max-heap by index: negate so heapq (min-heap) gives highest index first
    left_wait  = [-i for i in range(n)]   # all workers start on left
    right_wait = []
    heapq.heapify(left_wait)

    left_work  = []   # (finish_putNew,   worker_idx)
    right_work = []   # (finish_pickOld,  worker_idx)

    clock         = 0   # bridge free at this time
    dispatched_lr = 0   # total workers sent left → right
    dispatched_rl = 0   # total packages heading back (right → left trips started)
    ans           = 0   # time last putNew finishes

    # --- Main loop ---------------------------------------------------
    while dispatched_rl < k:

        # Release workers whose bank task finished by now
        while left_work  and left_work[0][0]  <= clock:
            heapq.heappush(left_wait,  -heapq.heappop(left_work)[1])
        while right_work and right_work[0][0] <= clock:
            heapq.heappush(right_wait, -heapq.heappop(right_work)[1])

        # If nobody is waiting, jump clock to next finish event
        if not left_wait and not right_wait:
            nxt = float('inf')
            if left_work:  nxt = min(nxt, left_work[0][0])
            if right_work: nxt = min(nxt, right_work[0][0])
            clock = nxt
            continue

        if right_wait:
            # Right → Left has priority: worker is carrying a package
            i          = -heapq.heappop(right_wait)
            clock     += time[i][2]                        # rightToLeft crossing
            put_done   = clock + time[i][3]                # will finish putNew at
            ans        = max(ans, put_done)                # track latest completion
            heapq.heappush(left_work, (put_done, i))
            dispatched_rl += 1

        elif left_wait and dispatched_lr < k:
            # Left → Right: fetch a package (only if we still need more)
            i          = -heapq.heappop(left_wait)
            clock     += time[i][0]                        # leftToRight crossing
            dispatched_lr += 1
            heapq.heappush(right_work, (clock + time[i][1], i))   # pickOld

        else:
            # Workers waiting on left but all k dispatched — advance to next event
            nxt = float('inf')
            if right_work: nxt = min(nxt, right_work[0][0])
            if left_work:  nxt = min(nxt, left_work[0][0])
            clock = nxt

    return ans


# 1 worker, 1 package: L→R(1) + pick(1) + R→L(2) + put(1) = done at t=5... 
# t=0→1 cross, t=1→2 pick, t=2→4 cross back, t=4→5 put
assert solution(1, 1, [[1,1,2,1]])  == 5

# 1 worker, 1 package, longer put: done at t=0+1+1+2+8 = 12
assert solution(1, 1, [[1,1,2,8]])  == 12

# 2 workers, 2 packages
assert solution(2, 2, [[1,9,1,8],[10,1,10,1]]) == 30

# 3 workers, 2 packages (only 2 workers needed)
assert solution(3, 2, [[1,1,2,1],[1,1,3,1],[1,1,2,1]]) == 8

print('All Pass!')